# 08 · 8-Bit KV Cache and Latency

**Project:** Silent Tool-Call Failures: How Quantization Degrades Function Calling in Small LLMs
**Goal (Update 13):** two measurements in one notebook.

1. **Stretch item, 8-bit KV cache:** the five FP16 models answer the 1,390 BFCL questions and the 175-question stress suite again, in free decoding, with one change: vLLM keeps the KV cache (the keys and values of the earlier tokens) in 8 bits (FP8 E4M3) instead of 16. Configs: `configs/<model>_fp16-kv8.yaml`. Weights, prompt, decoding and scorer are those of the FP16 runs, so any change in accuracy comes from the KV cache alone.
2. **Latency at batch size 1:** every config (the 25 model × precision combinations and the 5 with the 8-bit KV cache) answers the same 35 questions (5 per category, seed 42) one at a time, in free and in constrained decoding, with `src/latency.py`. It records the time to the first token, the time per output token and the time for the whole answer.

| Precision | Where the weights come from |
|---|---|
| FP16, FP16 with 8-bit KV cache, NF4 | the pinned Hugging Face model (NF4 is quantized by vLLM while loading) |
| INT8 | made here by `src/quantize.py` (under a minute per model) |
| GPTQ | downloaded from the private Hugging Face repositories made in Update 9, at a pinned commit |
| AWQ | downloaded from the private Hugging Face repositories made in Update 6, at a pinned commit |

**Checks first:** (a) the first model with the 8-bit KV cache answers 2 questions per category. The T4 has no FP8 hardware; vLLM 0.8.5 converts between FP16 and FP8 in its own CUDA code, so it should work, but only this check proves it. If it fails, the 8-bit KV-cache configs are skipped and the other latency runs still go ahead. (b) The first config answers 1 question per category with the latency script, in both modes. If that fails, the notebook stops.

**Fair timing:** one model runs at a time and nothing else runs in the session. Qwen3-8B in FP16 (with either KV cache) uses both T4s, as in the main runs; every other config uses one T4.

## How to run on Kaggle

As in notebooks 06 and 07, the work is split into two parts, so that one Kaggle session cannot lose all of it.

1. kaggle.com → **Create → New Notebook → File → Import Notebook**, then upload this file. Do this twice.
2. In the right panel, under **Settings**: **Accelerator → GPU T4 x2** and **Internet → On**.
3. **Add-ons → Secrets:** switch on `HF_TOKEN` for each notebook (needed for Llama and for the private checkpoints).
4. In the settings cell of the second notebook, set `PART = "B"`.
5. Click **Save Version → Save & Run All (Commit)** in both.

**Time:** not measured yet. Each config is loaded once for the latency runs (30 loads), and each FP16 model with the 8-bit KV cache answers the 1,390 + 175 questions once. A rough guess is 2.5 to 3.5 hours for part A and 2 to 3 hours for part B, plus about 20 minutes of installation.
**Output:** `update13_results_A.zip` and `update13_results_B.zip`, with the raw answers, scores, latency files, logs and `kv_latency_A.md` / `kv_latency_B.md`.

## 1. Settings
Edit this cell only.

In [1]:
import os

REPO_URL = "https://github.com/shawalkabirchy/quantization-functional-call"
REPO_REF = "main"             # branch, tag or commit of the code to run

# Two runs, one per part: "A" and "B". The parts can run at the same time, in two notebooks.
PART = "A"
MODELS_OF_PART = {"A": ["qwen3-1.7b", "llama-3.2-1b", "llama-3.2-3b"], "B": ["qwen3-4b", "qwen3-8b"]}
PRECISIONS = ["fp16", "fp16-kv8", "int8", "nf4", "gptq", "awq"]
# Grouped by model, so each model is downloaded once
CONFIGS = [f"configs/{m}_{p}.yaml" for m in MODELS_OF_PART[PART] for p in PRECISIONS]

RUN_KV = True                 # stretch item: the *_fp16-kv8 configs (False: skip them completely, latency too)
KV_RUNS = [("non_live", "free"), ("stress", "free")]   # (questions, decoding): BFCL's 1,390, then the stress suite
RUN_LATENCY = True            # latency at batch size 1 for every config
LATENCY_PER_CATEGORY = 5      # timed questions per BFCL category (35 in all), the same for every config
CHECK_LIMIT = 2               # questions per category in the 8-bit KV-cache check
REPORT_NAME = f"kv_latency_{PART}.md"
RUN_TIMEOUT_MIN = 90          # an accuracy run that takes longer than this is stopped, and the next step starts
LATENCY_TIMEOUT_MIN = 60      # the same for one config's latency measurement
QUANTIZE_TIMEOUT_MIN = 30     # the same for making an INT8 checkpoint
DELETE_MODEL_AFTER_RUN = True # frees disk space: deletes each checkpoint after its runs, each model after its last run

# 4-bit checkpoints made in Updates 6 (AWQ) and 9 (GPTQ), private on Hugging Face, pinned to the commit used there
HF_CHECKPOINT_USER = "shawalkabir"
CHECKPOINT_COMMITS = {
    "qwen3-1.7b_gptq": "7803089b341f86527b23c2a49f9bd4cc95662352",
    "qwen3-4b_gptq": "a82f18018d851518178ef4ce6cae34118c55da21",
    "qwen3-8b_gptq": "bec7853f0aae7bd4b51c30b6301f386caf584f53",
    "llama-3.2-1b_gptq": "b14742c0f416130b2e2ce2c595be390f15463c2b",
    "llama-3.2-3b_gptq": "59872d087ea389a4a30197553d7fe1b6d82770e7",
    "qwen3-1.7b_awq": "3cc45f1ddb6c1fcd967a26f5b649556d884360f2",
    "qwen3-4b_awq": "e8a7317cb3f6f927bea3e442b00e487d4a2d9ec1",
    "qwen3-8b_awq": "6f1d2a83ad23b41709e98366253f7f86fcf0a9ea",
    "llama-3.2-1b_awq": "e8cd172864e551542651600ebfbcbfc667328c90",
    "llama-3.2-3b_awq": "6f9fd9e999f07abdfe75085e85e0f9f784cad923",
}

BFCL_VERSION = "2026.3.23"    # pinned for reproducibility
BITSANDBYTES_VERSION = "0.45.5"  # same vLLM environment as notebooks 04-07
ACCELERATE_VERSION = "1.6.0"
WORK = "/kaggle/working"
REPO = f"{WORK}/repo"                         # code and results (kept as notebook output)
VENV = os.path.expanduser("~/bfcl-venv")      # vLLM + BFCL environment (not kept)
CHECKPOINT_DIR = os.path.expanduser("~/checkpoints")   # INT8, GPTQ and AWQ checkpoints (not kept: they are large)
os.environ["CHECKPOINT_DIR"] = CHECKPOINT_DIR           # read by src/quantize.py, src/run_eval.py and src/latency.py


def stem_of(config):
    return os.path.splitext(os.path.basename(config))[0]


def run_id_of(config, questions, mode):
    return f"{stem_of(config)}_{mode}" + ("_stress" if questions == "stress" else "")

## 2. Check the GPU, internet and the Hugging Face token

In [2]:
import json
import shutil
import subprocess
import time
from datetime import date

import requests

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if smi.returncode != 0:
    raise RuntimeError("No GPU found. Set Settings -> Accelerator -> GPU T4 x2, then run again.")
print("GPUs:\n" + smi.stdout)

try:
    requests.get("https://pypi.org/simple/", timeout=15).raise_for_status()
    print("Internet: OK")
except Exception as e:
    raise RuntimeError("No internet. Set Settings -> Internet -> On (needs a phone-verified Kaggle account).") from e

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN: loaded from Kaggle Secrets")
except Exception:
    raise RuntimeError("HF_TOKEN is not set. Llama 3.2 and the private checkpoints need it (Add-ons -> Secrets).")

GPUs:
0, Tesla T4, 15360 MiB, 580.178.04
1, Tesla T4, 15360 MiB, 580.178.04

Internet: OK
HF_TOKEN: loaded from Kaggle Secrets


## 3. Get the code and check access to every model
Clones the repository, then downloads one small file of each model at the pinned revision. This finds a missing token or missing Llama approval now, not after an hour of running.

In [3]:
import yaml
from huggingface_hub import hf_hub_download

shutil.rmtree(REPO, ignore_errors=True)
!git clone --quiet {REPO_URL} {REPO}
!git -C {REPO} checkout --quiet {REPO_REF}
!git -C {REPO} log -1 --format="Code version: %h %s (%ci)"

cfgs = {}
problems = []
for config in CONFIGS:
    with open(f"{REPO}/{config}") as f:
        cfgs[config] = yaml.safe_load(f)
    model, revision = cfgs[config]["model"], str(cfgs[config]["revision"])
    try:
        hf_hub_download(model, "config.json", revision=revision, token=os.environ.get("HF_TOKEN"))
        print(f"  access OK  {model}")
    except Exception as e:
        problems.append(f"{model}: {type(e).__name__}: {str(e)[:200]}")
if problems:
    print("\n".join(problems))
    raise RuntimeError(
        "Some models cannot be downloaded. For Llama: add HF_TOKEN under Add-ons -> Secrets, switch it on for "
        "this notebook, and check that https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct says you have access."
    )

# The stress suite comes with the code: 7 groups of 25 questions
for group in ["stress_plain", "stress_nested", "stress_enum", "stress_number", "stress_optional", "stress_unicode",
              "stress_menu"]:
    with open(f"{REPO}/data/stress_suite/{group}.json", encoding="utf-8") as f:
        n = sum(1 for _ in f)
    assert n == 25, (group, n)
print("Stress suite: 7 groups of 25 questions")

Code version: cdd3e35 Update 13 prep: 8-bit KV-cache configs, latency script, notebook 08 (2026-11-16 22:55:10 +0600)


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

  access OK  Qwen/Qwen3-1.7B
  access OK  Qwen/Qwen3-1.7B
  access OK  Qwen/Qwen3-1.7B
  access OK  Qwen/Qwen3-1.7B
  access OK  Qwen/Qwen3-1.7B
  access OK  Qwen/Qwen3-1.7B


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

  access OK  meta-llama/Llama-3.2-1B-Instruct
  access OK  meta-llama/Llama-3.2-1B-Instruct
  access OK  meta-llama/Llama-3.2-1B-Instruct
  access OK  meta-llama/Llama-3.2-1B-Instruct
  access OK  meta-llama/Llama-3.2-1B-Instruct
  access OK  meta-llama/Llama-3.2-1B-Instruct


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

  access OK  meta-llama/Llama-3.2-3B-Instruct
  access OK  meta-llama/Llama-3.2-3B-Instruct
  access OK  meta-llama/Llama-3.2-3B-Instruct
  access OK  meta-llama/Llama-3.2-3B-Instruct
  access OK  meta-llama/Llama-3.2-3B-Instruct
  access OK  meta-llama/Llama-3.2-3B-Instruct
Stress suite: 7 groups of 25 questions


## 4. Install the vLLM environment (about 1-5 minutes)
Exactly the same environment as notebooks 04-07: a separate Python 3.11 environment with `bfcl-eval[oss_eval_vllm]` (vLLM 0.8.5) and the same pins. No GPTQModel environment is needed here: the GPTQ and AWQ checkpoints are downloaded, and INT8 is made with bitsandbytes in this environment.

In [4]:
!pip install -q uv
shutil.rmtree(VENV, ignore_errors=True)
!uv venv {VENV} --python 3.11
!uv pip install --python {VENV}/bin/python "bfcl-eval[oss_eval_vllm]=={BFCL_VERSION}" "transformers==4.51.3" "openai>=1.86.0,<2" "fastapi<0.116" "pydantic<2.12" "setuptools>=77.0.3,<80" soundfile "bitsandbytes=={BITSANDBYTES_VERSION}" "accelerate=={ACCELERATE_VERSION}"

CHECK = (
    "import vllm, bfcl_eval, yaml, xgrammar, triton, torch, bitsandbytes, accelerate; "
    "from importlib.metadata import version; "
    "print('vllm', version('vllm'), '| xgrammar', version('xgrammar'), '| torch', version('torch'), "
    "'| bitsandbytes', version('bitsandbytes'), '| accelerate', version('accelerate')); "
    "print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"
)
r = subprocess.run([f"{VENV}/bin/python", "-c", CHECK], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-5000:])
    raise RuntimeError("Environment check failed (see above). Check the output of the install step.")

freeze = subprocess.run(["uv", "pip", "freeze", "--python", f"{VENV}/bin/python"], capture_output=True, text=True)
with open(f"{WORK}/requirements-freeze.txt", "w") as f:
    f.write(freeze.stdout)
print(f"Saved {len(freeze.stdout.splitlines())} package versions to {WORK}/requirements-freeze.txt")

Using CPython 3.11.18
Creating virtual environment at: /root/bfcl-venv
Activate with: source /root/bfcl-venv/bin/activate
Using Python 3.11.18 environment at: /root/bfcl-venv
Resolved 215 packages in 2.35s
Prepared 215 packages in 4m 07s
Installed 215 packages in 508ms
 + accelerate==1.6.0
 + agent-detector==2.0.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + airportsdata==20260905
 + annotated-doc==0.0.6
 + annotated-types==0.8.0
 + anthropic==2.9.0
 + anyio==4.16.0
 + argcomplete==3.7.2
 + astor==0.8.1
 + attrs==26.1.0
 + beautifulsoup4==4.15.0
 + bfcl-eval==2026.3.23
 + bitsandbytes==0.45.5
 + black==26.10.1
 + blake3==1.0.11
 + boto3==1.44.3
 + botocore==1.44.3
 + cachetools==8.0.0
 + certifi==2026.10.23
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + click==8.5.1
 + cloudpickle==3.1.2
 + cohere==5.18.0
 + compressed-tensors==0.9.3
 + cryptography==50.0.2
 + cupy-cuda12x==13.6.0
 + dashscope==2.1.0
 + datamodel-code-generator==0.25.7
 + deprecated==1.3.1
 + de

## 5. Checks before the long runs
(a) The 8-bit KV cache on the T4: the first model answers 2 questions per category; the right/wrong results are compared with its FP16 run in the repository. If this fails, the 8-bit KV-cache configs are skipped. (b) The latency script on the first config, 1 question per category in both modes. If this fails, the notebook stops. The answers go to `check/`, apart from the real results.

In [5]:
def tail(path, n=25):
    with open(path, errors="replace") as f:
        return "".join(f.readlines()[-n:])


def run_logged(args, log_path, timeout_min, venv=VENV):
    """Runs a script of the repository in an environment; returns its exit code, or None after a timeout."""
    with open(log_path, "w") as log:
        try:
            return subprocess.run([f"{venv}/bin/python", *args], cwd=REPO, stdout=log, stderr=subprocess.STDOUT,
                                  timeout=timeout_min * 60).returncode
        except subprocess.TimeoutExpired:
            print(f"  stopped after {timeout_min} minutes")
            return None


def gpu_memory_used_mib():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout
    return [int(x) for x in out.split()]


def wait_for_free_gpus(limit_mib=1000, wait_s=120):
    """A finished run should leave the GPUs empty; wait briefly if they are not yet."""
    start = time.time()
    while max(gpu_memory_used_mib()) > limit_mib and time.time() - start < wait_s:
        time.sleep(5)
    used = gpu_memory_used_mib()
    if max(used) > limit_mib:
        print(f"  WARNING: GPU memory still in use before this run: {used} MiB")


def make_checkpoint(config, stem, log_path):
    """Downloads a GPTQ or AWQ checkpoint at its pinned commit, or makes the INT8 one. Returns True if it worked."""
    method = cfgs[config]["quantize"]["method"]
    if method in ("gptq", "awq"):
        from huggingface_hub import snapshot_download
        repo_id = f"{HF_CHECKPOINT_USER}/{cfgs[config]['model'].split('/')[-1]}-{method.upper()}-W4A16-g128"
        with open(log_path, "w") as log:
            try:
                if not os.path.exists(f"{CHECKPOINT_DIR}/{stem}/quantize_info.json"):
                    snapshot_download(repo_id, revision=CHECKPOINT_COMMITS[stem], local_dir=f"{CHECKPOINT_DIR}/{stem}",
                                      token=os.environ.get("HF_TOKEN"))
                log.write(f"Using {repo_id} at {CHECKPOINT_COMMITS[stem][:7]}\n")
                return True
            except Exception as e:
                log.write(f"Could not download {repo_id}: {type(e).__name__}: {str(e)[:300]}\n")
                return False
    return run_logged(["src/quantize.py", "--config", config], log_path, QUANTIZE_TIMEOUT_MIN) == 0


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]


CHECK_DIR = f"{WORK}/check"
shutil.rmtree(CHECK_DIR, ignore_errors=True)
first_model = MODELS_OF_PART[PART][0]

# (a) The 8-bit KV cache on the T4
KV_OK = False
if RUN_KV:
    config = f"configs/{first_model}_fp16-kv8.yaml"
    print(f"=== check {stem_of(config)}: started {time.strftime('%H:%M:%S')} UTC", flush=True)
    wait_for_free_gpus()
    log_path = f"{WORK}/check_{stem_of(config)}.log"
    ok = run_logged(["src/run_eval.py", "--config", config, "--mode", "free", "--limit", str(CHECK_LIMIT),
                     "--results-dir", CHECK_DIR, "--overwrite"], log_path, 30) == 0
    run_id = f"{run_id_of(config, 'non_live', 'free')}_limit{CHECK_LIMIT}"
    if ok:
        r = subprocess.run([f"{VENV}/bin/python", "src/score.py", f"{CHECK_DIR}/raw/{run_id}.jsonl"],
                           cwd=REPO, capture_output=True, text=True)
        ok = r.returncode == 0
        print(r.stdout[-1500:] if ok else r.stderr[-3000:])
    if ok:
        KV_OK = True
        fp16 = {s["id"]: s["valid"] for s in read_jsonl(f"{REPO}/results/scores/{first_model}_fp16_free.jsonl")}
        kv8 = read_jsonl(f"{CHECK_DIR}/scores/{run_id}.jsonl")
        same = sum(s["valid"] == fp16[s["id"]] for s in kv8)
        print(f"Same right/wrong result as the FP16 run on {same} of {len(kv8)} questions")
        for row in read_jsonl(f"{CHECK_DIR}/raw/{run_id}.jsonl")[:2]:
            print(f"  {row['id']}: {row['output'][:200]!r}")
        print("8-bit KV cache check: OK")
    else:
        print(f"End of {log_path}:\n{tail(log_path, 40)}")
        print("8-bit KV cache check FAILED: the 8-bit KV-cache configs are skipped; the other latency runs still go ahead.")

# (b) The latency script
if RUN_LATENCY:
    config = CONFIGS[0]
    print(f"\n=== check latency {stem_of(config)}: started {time.strftime('%H:%M:%S')} UTC", flush=True)
    wait_for_free_gpus()
    log_path = f"{WORK}/check_latency_{stem_of(config)}.log"
    ok = run_logged(["src/latency.py", "--config", config, "--per-category", "1", "--warmup", "1",
                     "--results-dir", CHECK_DIR, "--overwrite"], log_path, 30) == 0
    if not ok:
        print(f"End of {log_path}:\n{tail(log_path, 40)}")
        raise RuntimeError("The latency check failed; stopping before the long runs.")
    print(tail(log_path, 6))
print("\nChecks done.")

=== check qwen3-1.7b_fp16-kv8: started 17:46:16 UTC

qwen3-1.7b_fp16-kv8_free_limit2  (14 questions)
  category               correct  accuracy   parsed
  simple_python              1/2     50.00    50.00
  simple_java                1/2     50.00    50.00
  simple_javascript          1/2     50.00   100.00
  multiple                   2/2    100.00   100.00
  parallel                   2/2    100.00   100.00
  parallel_multiple          1/2     50.00    50.00
  irrelevance                2/2    100.00   100.00
  saved /kaggle/working/check/scores/qwen3-1.7b_fp16-kv8_free_limit2.jsonl and /kaggle/working/check/summary.csv

Same right/wrong result as the FP16 run on 14 of 14 questions
  simple_python_0: '[{"name": "calculate_triangle_area", "arguments": {"base": 10, "height": 5}}]'
  simple_python_1: '[{"name": "math.factorial", "arguments": {"number": 5}}}]'
8-bit KV cache check: OK

=== check latency qwen3-1.7b_fp16: started 17:47:49 UTC

constrained: first token 601.7 ms, 54.1 tokens

## 6. Run every config (a few hours)
One config at a time: the checkpoint (only INT8, GPTQ and AWQ need one); for the configs with the 8-bit KV cache, `run_eval.py` and `score.py` on BFCL's questions and on the stress suite; then `latency.py`. Every step's full log is saved. If a step fails, the notebook prints the end of its log and goes on with the next one.

In [6]:
def answer_and_score(config, questions, mode):
    """One accuracy run (run_eval.py) and its scoring (score.py). Returns the status."""
    run_id = run_id_of(config, questions, mode)
    print(f"\n=== {run_id}: started {time.strftime('%H:%M:%S')} UTC", flush=True)
    wait_for_free_gpus()
    log_path = f"{WORK}/run_{run_id}.log"
    run_logged(["src/run_eval.py", "--config", config, "--mode", mode, "--categories", questions, "--overwrite"],
               log_path, RUN_TIMEOUT_MIN)
    if not os.path.exists(f"{REPO}/results/meta/{run_id}.json"):
        print(f"  FAILED. End of {log_path}:\n{tail(log_path)}")
        return "FAILED"
    print("  answers: " + next((l.strip() for l in tail(log_path, 10).splitlines() if l.startswith("Done:")), ""))
    r = subprocess.run([f"{VENV}/bin/python", "src/score.py", f"results/raw/{run_id}.jsonl"],
                       cwd=REPO, capture_output=True, text=True)
    print(r.stdout[-1500:])
    if r.returncode != 0:
        print(r.stderr[-3000:])
        return "SCORING FAILED"
    return "OK"


def measure_latency(config):
    """One config's latency at batch size 1 (latency.py). Returns the status."""
    stem = stem_of(config)
    print(f"\n=== latency {stem}: started {time.strftime('%H:%M:%S')} UTC", flush=True)
    wait_for_free_gpus()
    log_path = f"{WORK}/latency_{stem}.log"
    run_logged(["src/latency.py", "--config", config, "--per-category", str(LATENCY_PER_CATEGORY), "--overwrite"],
               log_path, LATENCY_TIMEOUT_MIN)
    if not os.path.exists(f"{REPO}/results/latency/{stem}.json"):
        print(f"  FAILED. End of {log_path}:\n{tail(log_path)}")
        return "FAILED"
    print("\n".join("  " + l for l in tail(log_path, 8).splitlines() if l.startswith(("free:", "constrained:"))))
    return "OK"


status, minutes = {}, {}
last_config_of = {cfgs[c]["model"]: c for c in CONFIGS}   # each model is deleted after its last config
for config in CONFIGS:
    cfg, stem = cfgs[config], stem_of(config)
    jobs = []                 # (name, questions, mode); questions None = latency
    if cfg["precision"] == "fp16-kv8":
        if KV_OK:             # without a working 8-bit KV cache these configs are skipped completely
            jobs += [(run_id_of(config, questions, mode), questions, mode) for questions, mode in KV_RUNS]
    if RUN_LATENCY and (cfg["precision"] != "fp16-kv8" or KV_OK):
        jobs.append((f"latency {stem}", None, None))
    ok = True
    if jobs and "quantize" in cfg:
        print(f"\n=== {stem}: checkpoint, started {time.strftime('%H:%M:%S')} UTC", flush=True)
        wait_for_free_gpus()
        log_path = f"{WORK}/quantize_{stem}.log"
        ok = make_checkpoint(config, stem, log_path)
        print("  " + tail(log_path, 1).strip() if ok else f"  checkpoint FAILED. End of {log_path}:\n{tail(log_path)}")
    for name, questions, mode in jobs:
        if not ok:
            status[name] = "CHECKPOINT FAILED"
            continue
        start = time.time()
        status[name] = measure_latency(config) if questions is None else answer_and_score(config, questions, mode)
        minutes[name] = (time.time() - start) / 60
        print(f"  {status[name]} after {minutes[name]:.1f} min")

    if DELETE_MODEL_AFTER_RUN:
        shutil.rmtree(f"{CHECKPOINT_DIR}/{stem}", ignore_errors=True)
        if last_config_of[cfg["model"]] == config:
            shutil.rmtree(os.path.expanduser(f"~/.cache/huggingface/hub/models--{cfg['model'].replace('/', '--')}"),
                          ignore_errors=True)

print("\nSummary:")
for name, s in status.items():
    took = f"  ({minutes[name]:.1f} min)" if name in minutes else ""
    print(f"  {name:40s} {s}{took}")
print(f"  total: {sum(minutes.values()) / 60:.1f} hours of runs")
if any(s != "OK" for s in status.values()):
    print("Some steps did not finish. The results of the others are still reported and packaged below.")


=== latency qwen3-1.7b_fp16: started 17:49:06 UTC
  free: first token 76.9 ms, 52.9 tokens/s, whole answer 0.795 s (medians); 61.5 tokens on average
  constrained: first token 583.4 ms, 49.6 tokens/s, whole answer 1.408 s (medians); 61.6 tokens on average
  OK after 2.9 min

=== qwen3-1.7b_fp16-kv8_free: started 17:52:01 UTC
  answers: Done: 1390 answers in 326 s (217.4 output tokens/s); finish reasons {'stop': 1390}

qwen3-1.7b_fp16-kv8_free  (1390 questions)
  category               correct  accuracy   parsed
  simple_python          316/400     79.00    85.00
  simple_java             66/100     66.00    99.00
  simple_javascript        33/50     66.00    96.00
  multiple               166/200     83.00    92.00
  parallel               150/200     75.00    90.00
  parallel_multiple      118/200     59.00    71.00
  irrelevance            189/240     78.75    98.75
  simple_ast                         70.33         
  non_live_overall                   71.83         
  saved /kaggl

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

  Using shawalkabir/Qwen3-1.7B-GPTQ-W4A16-g128 at 7803089

=== latency qwen3-1.7b_gptq: started 18:19:48 UTC
  free: first token 85.7 ms, 101.4 tokens/s, whole answer 0.485 s (medians); 66.8 tokens on average
  constrained: first token 604.3 ms, 100.4 tokens/s, whole answer 1.027 s (medians); 67.0 tokens on average
  OK after 2.3 min

=== qwen3-1.7b_awq: checkpoint, started 18:22:04 UTC


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

... (9 outputs left out)

  Using shawalkabir/Llama-3.2-3B-Instruct-GPTQ-W4A16-g128 at 59872d0

=== latency llama-3.2-3b_gptq: started 19:43:24 UTC
  free: first token 191.3 ms, 76.6 tokens/s, whole answer 0.855 s (medians); 78.3 tokens on average
  constrained: first token 527.0 ms, 77.6 tokens/s, whole answer 0.975 s (medians); 48.5 tokens on average
  OK after 2.6 min

=== llama-3.2-3b_awq: checkpoint, started 19:45:58 UTC


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

  Using shawalkabir/Llama-3.2-3B-Instruct-AWQ-W4A16-g128 at 6f9fd9e

=== latency llama-3.2-3b_awq: started 19:46:02 UTC
  free: first token 291.2 ms, 43.4 tokens/s, whole answer 1.227 s (medians); 61.8 tokens on average
  constrained: first token 533.8 ms, 46.8 tokens/s, whole answer 1.33 s (medians); 54.7 tokens on average
  OK after 3.4 min

Summary:
  latency qwen3-1.7b_fp16                  OK  (2.9 min)
  qwen3-1.7b_fp16-kv8_free                 OK  (6.5 min)
  qwen3-1.7b_fp16-kv8_free_stress          OK  (1.9 min)
  latency qwen3-1.7b_fp16-kv8              OK  (3.3 min)
  latency qwen3-1.7b_int8                  OK  (11.8 min)
  latency qwen3-1.7b_nf4                   OK  (4.0 min)
  latency qwen3-1.7b_gptq                  OK  (2.3 min)
  latency qwen3-1.7b_awq                   OK  (2.4 min)
  latency llama-3.2-1b_fp16                OK  (2.6 min)
  llama-3.2-1b_fp16-kv8_free               OK  (6.6 min)
  llama-3.2-1b_fp16-kv8_free_stress        OK  (2.9 min)
  latency llama-3

## 7. Tables
(a) Accuracy with the 8-bit KV cache next to the FP16 runs already in the repository (same weights). (b) Latency at batch size 1: medians over the 35 questions.

In [7]:
import pandas as pd

LABELS = {"fp16": "FP16", "fp16-kv8": "FP16, 8-bit KV", "int8": "INT8", "nf4": "NF4", "gptq": "GPTQ", "awq": "AWQ"}
BFCL_COLUMNS = {"simple_ast": "Simple", "multiple": "Multiple", "parallel": "Parallel",
                "parallel_multiple": "Par.-mult.", "irrelevance": "Irrelevance", "non_live_overall": "Overall"}
STRESS_COLUMNS = {"stress_plain": "Plain", "stress_nested": "Nested", "stress_enum": "Enum", "stress_number": "Number",
                  "stress_optional": "Optional", "stress_unicode": "Unicode", "stress_menu": "Menu"}
summary = pd.read_csv(f"{REPO}/results/summary.csv")


def accuracy_row(name, label, run_id, columns):
    s = summary[summary["run_id"] == run_id].set_index("category")
    if s.empty:
        return None
    return {"Model": name, "KV cache": label,
            **{col: float(s.loc[c, "accuracy"]) if c in s.index else None for c, col in columns.items()}}


bfcl_rows, stress_rows = [], []
for config in CONFIGS:
    if cfgs[config]["precision"] != "fp16-kv8":
        continue
    name, fp16_stem = cfgs[config]["model"].split("/")[-1], stem_of(config).replace("_fp16-kv8", "_fp16")
    for rows, questions, columns in [(bfcl_rows, "non_live", BFCL_COLUMNS), (stress_rows, "stress", STRESS_COLUMNS)]:
        suffix = "_stress" if questions == "stress" else ""
        base = accuracy_row(name, "FP16", f"{fp16_stem}_free{suffix}", columns)
        kv = accuracy_row(name, "8-bit", run_id_of(config, questions, "free"), columns)
        if kv is not None:    # FP16 rows only next to a finished 8-bit run
            rows += [r for r in (base, kv) if r is not None]
bfcl, stress = pd.DataFrame(bfcl_rows), pd.DataFrame(stress_rows)
if len(bfcl):
    print("(a) BFCL accuracy (%), free decoding")
    display(bfcl)
if len(stress):
    print("Stress suite accuracy (%), free decoding")
    display(stress)

latency_rows = []
for config in CONFIGS:
    path = f"{REPO}/results/latency/{stem_of(config)}.json"
    if not os.path.exists(path):
        continue
    with open(path) as f:
        meta = json.load(f)
    for mode, s in meta["summary"].items():
        latency_rows.append({
            "Model": meta["model"].split("/")[-1], "Precision": LABELS[meta["precision"]],
            "GPUs": meta["tensor_parallel_size"], "Mode": mode,
            "First token (ms)": s["first_token_ms_median"], "Tokens/s": s["output_tokens_per_second"],
            "Whole answer (s)": s["total_s_median"], "Output tokens (mean)": s["output_tokens_mean"],
            "Cut off": s["finish_reasons"].get("length", 0),
            "Weights per GPU (GiB)": meta["memory"]["model_weights_gib"],
            "KV cache blocks": meta["memory"]["kv_cache_blocks"],
        })
latency = pd.DataFrame(latency_rows)
if len(latency):
    print("(b) Latency at batch size 1 (medians over the questions)")
    display(latency)
bfcl.to_csv(f"{WORK}/kv_bfcl_{PART}.csv", index=False)
stress.to_csv(f"{WORK}/kv_stress_{PART}.csv", index=False)
latency.to_csv(f"{WORK}/latency_{PART}.csv", index=False)

(a) BFCL accuracy (%), free decoding


,Model,KV cache,Simple,Multiple,Parallel,Par.-mult.,Irrelevance,Overall
0,Qwen3-1.7B,FP16,71.75,87.5,71.5,59.5,77.92,72.56
1,Qwen3-1.7B,8-bit,70.33,83.0,75.0,59.0,78.75,71.83
2,Llama-3.2-1B-Instruct,FP16,41.58,38.5,41.0,24.5,82.92,36.40
3,Llama-3.2-1B-Instruct,8-bit,38.92,40.0,39.0,22.0,82.50,34.98
4,Llama-3.2-3B-Instruct,FP16,26.00,30.0,42.5,27.0,91.25,31.38
5,Llama-3.2-3B-Instruct,8-bit,26.08,34.0,44.0,21.5,90.83,31.40


Stress suite accuracy (%), free decoding


,Model,KV cache,Plain,Nested,Enum,Number,Optional,Unicode,Menu
0,Qwen3-1.7B,FP16,100.0,96.0,88.0,36.0,100.0,96.0,100.0
1,Qwen3-1.7B,8-bit,88.0,88.0,92.0,32.0,96.0,92.0,100.0
2,Llama-3.2-1B-Instruct,FP16,8.0,44.0,20.0,12.0,56.0,16.0,12.0
3,Llama-3.2-1B-Instruct,8-bit,8.0,44.0,20.0,12.0,48.0,16.0,8.0
4,Llama-3.2-3B-Instruct,FP16,64.0,48.0,44.0,36.0,28.0,20.0,76.0
5,Llama-3.2-3B-Instruct,8-bit,64.0,44.0,44.0,36.0,32.0,16.0,76.0


(b) Latency at batch size 1 (medians over the questions)


,Model,Precision,GPUs,Mode,First token (ms),Tokens/s,Whole answer (s),Output tokens (mean),Cut off,Weights per GPU (GiB),KV cache blocks
0,Qwen3-1.7B,FP16,1,free,76.9,52.9,0.795,61.5,0,3.2152,4931
1,Qwen3-1.7B,FP16,1,constrained,583.4,49.6,1.408,61.6,0,3.2152,4931
2,Qwen3-1.7B,"FP16, 8-bit KV",1,free,82.6,40.6,0.997,61.1,0,3.2152,9862
3,Qwen3-1.7B,"FP16, 8-bit KV",1,constrained,599.6,41.3,1.531,61.2,0,3.2152,9862
4,Qwen3-1.7B,INT8,1,free,354.0,7.3,5.461,61.0,0,1.9027,5690
5,Qwen3-1.7B,INT8,1,constrained,873.0,7.2,6.245,60.7,0,1.9027,5690
6,Qwen3-1.7B,NF4,1,free,566.7,47.4,1.499,62.8,0,1.2674,6070
7,Qwen3-1.7B,NF4,1,constrained,838.7,46.3,1.687,60.4,0,1.2674,6070
8,Qwen3-1.7B,GPTQ,1,free,85.7,101.4,0.485,66.8,0,1.2721,6066
9,Qwen3-1.7B,GPTQ,1,constrained,604.3,100.4,1.027,67.0,0,1.2721,6066


## 8. Write the report and package the results

In [8]:
import glob


def md_table(df):
    def fmt(v):
        if v is None or (isinstance(v, float) and v != v):
            return ""
        return f"{v:.2f}" if isinstance(v, float) else str(v)
    head = "| " + " | ".join(df.columns) + " |"
    sep = "|" + "|".join("---" if i == 0 else "---:" for i in range(len(df.columns))) + "|"
    return [head, sep] + ["| " + " | ".join(fmt(v) for v in row) + " |" for row in df.itertuples(index=False)]


commit = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
lines = [f"# 8-bit KV cache and latency, part {PART} (Update 13)", "", f"**Date:** {date.today()}",
         f"**Code:** commit `{commit}`; the same models, prompts, decoding settings and scorer as the main runs.", ""]
if len(bfcl):
    lines += ["## BFCL accuracy (%), free decoding: FP16 vs. 8-bit KV cache", ""] + md_table(bfcl) + [""]
if len(stress):
    lines += ["## Stress-suite accuracy (%), free decoding: FP16 vs. 8-bit KV cache", ""] + md_table(stress) + [""]
if not len(bfcl) and not len(stress):
    lines += ["## 8-bit KV cache", "", "No KV-cache results (the check failed or RUN_KV was off).", ""]
if len(latency):
    lines += ["## Latency at batch size 1 (medians over 35 questions)", ""] + md_table(latency) + [""]
failed = [name for name, s in status.items() if s != "OK"]
lines += ["## Notes", "", f"- Did not finish: {', '.join(failed)}" if failed else "- All steps finished.", ""]
report = "\n".join(lines) + "\n"
with open(f"{WORK}/{REPORT_NAME}", "w") as f:
    f.write(report)
print(report)

pack = f"{WORK}/update13_results_{PART}"
shutil.rmtree(pack, ignore_errors=True)
shutil.copytree(f"{REPO}/results", f"{pack}/results", ignore=shutil.ignore_patterns("bfcl_repro", "figures"))
if os.path.exists(CHECK_DIR):
    shutil.copytree(CHECK_DIR, f"{pack}/check")
for name in [REPORT_NAME, f"kv_bfcl_{PART}.csv", f"kv_stress_{PART}.csv", f"latency_{PART}.csv",
             "requirements-freeze.txt"]:
    if os.path.exists(f"{WORK}/{name}"):
        shutil.copy(f"{WORK}/{name}", pack)
for log_path in sorted(glob.glob(f"{WORK}/*.log")):
    shutil.copy(log_path, pack)
archive = shutil.make_archive(pack, "zip", pack)
print("Saved", archive)

# 8-bit KV cache and latency, part A (Update 13)

**Date:** 2026-11-16
**Code:** commit `cdd3e35`; the same models, prompts, decoding settings and scorer as the main runs.

## BFCL accuracy (%), free decoding: FP16 vs. 8-bit KV cache

| Model | KV cache | Simple | Multiple | Parallel | Par.-mult. | Irrelevance | Overall |
|---|---:|---:|---:|---:|---:|---:|---:|
| Qwen3-1.7B | FP16 | 71.75 | 87.50 | 71.50 | 59.50 | 77.92 | 72.56 |
| Qwen3-1.7B | 8-bit | 70.33 | 83.00 | 75.00 | 59.00 | 78.75 | 71.83 |
| Llama-3.2-1B-Instruct | FP16 | 41.58 | 38.50 | 41.00 | 24.50 | 82.92 | 36.40 |
| Llama-3.2-1B-Instruct | 8-bit | 38.92 | 40.00 | 39.00 | 22.00 | 82.50 | 34.98 |
| Llama-3.2-3B-Instruct | FP16 | 26.00 | 30.00 | 42.50 | 27.00 | 91.25 | 31.38 |
| Llama-3.2-3B-Instruct | 8-bit | 26.08 | 34.00 | 44.00 | 21.50 | 90.83 | 31.40 |

## Stress-suite accuracy (%), free decoding: FP16 vs. 8-bit KV cache

| Model | KV cache | Plain | Nested | Enum | Number | Optional | Unicode | Menu |
|---|---:|---:|

## After the run: what goes into the repository

1. Download **`update13_results_A.zip`** and **`update13_results_B.zip`** from the Output tab of each saved version, and each notebook with its outputs.
2. Merge the two parts: `results/latency/`, and for the 8-bit KV cache `results/meta/`, `results/scores/` and the new rows of `results/summary.csv`; the raw answers in `results/raw/` stay out of git.
3. Compare with FP16 (significance test as in Update 8, failure classes as in Update 7), check that the latency prompts equal the main runs' prompts, add the GPU time to `notes/compute_log.md`, any design choice to `notes/decisions.md`, and the entry to `UPDATES.md`, then commit and push.